# 01 — Data Audit

**India Packaged Food Label Impact & Market Analytics**

This notebook audits the two real public datasets and documents the raw→cleaned transformation. Nothing here is simulated — every number is read from the cleaned files and the metrics produced by the pipeline.

Sources: **Open Food Facts** (nutrition + official Nutri-Score, ODbL) and **BigBasket** (Indian catalogue + price).

In [1]:
import sys, json
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FINAL = ROOT / 'data' / 'final'
CLEAN = ROOT / 'data' / 'cleaned'
RAW = ROOT / 'data' / 'raw'
metrics = json.loads((ROOT / 'reports' / 'computed_metrics.json').read_text())
pd.set_option('display.max_columns', 40)


## Cleaning audit — Open Food Facts
Raw vs cleaned row counts and the reasons rows were removed.

In [2]:
off = metrics['cleaning_off']
print('raw rows     :', off['raw_rows'])
print('cleaned rows :', off['cleaned_rows'])
print('full-row dupes removed :', off['removed_full_row_duplicates'])
print('name+brand dupes removed:', off['removed_name_brand_duplicates'])
print('impossible-value rows   :', off['removed_invalid_values'])
print('energy outliers flagged :', off['energy_statistical_outliers_flagged'])
print('classified %            :', off['pct_classified'])
print('grade coverage % (clean):', off['cleaned_grade_coverage_pct'])

raw rows     : 257387
cleaned rows : 239635
full-row dupes removed : 6797
name+brand dupes removed: 10256
impossible-value rows   : 699
energy outliers flagged : 804
classified %            : 75.47
grade coverage % (clean): 91.94


In [3]:
# impossible-value breakdown
pd.Series(off['invalid_value_detail']).sort_values(ascending=False)

sugars_exceed_carbs     546
energy_kj_100g           82
salt_g_100g              61
carbohydrates_g_100g      9
sugars_g_100g             5
fat_g_100g                2
proteins_g_100g           1
dtype: int64

## Cleaning audit — BigBasket (Indian market)

In [4]:
bb = metrics['cleaning_bigbasket']
for k, v in bb.items():
    print(f'{k:32s}: {v}')

raw_rows                        : 27555
raw_cols                        : 10
removed_non_food                : 14732
raw_missing_cells               : 4539
removed_missing_product_or_price: 1
removed_duplicates              : 615
cleaned_rows                    : 12207
cleaned_cols                    : 9
n_food_categories               : 7
n_sub_categories                : 55
n_brands                        : 1160
median_price_inr                : 140.25


## Load the cleaned data and inspect

In [5]:
food = pd.read_csv(CLEAN / 'cleaned_food_products.csv', low_memory=False)
print(food.shape)
food.head()

(239635, 13)


,product_name,brand_clean,brands,category,official_grade,energy_kj_100g,fat_g_100g,carbohydrates_g_100g,sugars_g_100g,proteins_g_100g,salt_g_100g,sodium_mg_100g,energy_outlier_flag
0,Banana Chips Sweetened (Whole),Unknown,NaN,"Chips, Crisps & Salty Snacks",D,2243.0,28.57,64.29,14.29,3.57,0.00000,0.000,0
1,Peanuts,Torn & Glasser,Torn & Glasser,"Nuts, Seeds & Dried Fruit",B,1941.0,17.86,60.71,17.86,17.86,0.63500,254.000,0
2,Organic Salted Nut Mix,Grizzlies,Grizzlies,Other / Unclassified,D,2540.0,57.14,17.86,3.57,17.86,1.22428,489.712,0
3,Organic Muesli,Daddy'S Muesli,Daddy's Muesli,Breakfast Cereals & Muesli,C,1833.0,18.75,57.81,15.62,14.06,0.13970,55.880,0
4,Zen Party Mix,Sunridge,Sunridge,Other / Unclassified,D,2230.0,36.67,36.67,3.33,16.67,1.60782,643.128,0


In [6]:
# missingness in the cleaned nutrition data
(food.isna().mean().mul(100).round(1)
     .sort_values(ascending=False).rename('pct_missing'))

official_grade          8.1
brands                  1.1
product_name            0.0
brand_clean             0.0
category                0.0
energy_kj_100g          0.0
fat_g_100g              0.0
carbohydrates_g_100g    0.0
sugars_g_100g           0.0
proteins_g_100g         0.0
salt_g_100g             0.0
sodium_mg_100g          0.0
energy_outlier_flag     0.0
Name: pct_missing, dtype: float64

In [7]:
# category coverage (engineered feature)
cov = (food['category'] != 'Other / Unclassified').mean() * 100
print(f'classified into a named category: {cov:.1f}%')
food['category'].value_counts().head(15)

classified into a named category: 75.5%


category
Other / Unclassified                    58782
Chocolate & Confectionery               25383
Sauces, Ketchup & Condiments            16438
Fruits & Vegetables (packaged/plain)    15495
Processed Meat & Seafood                12979
Cheese                                  11463
Instant Noodles, Pasta & Soups           9151
Cakes, Pastries & Sweet Bakery           8905
Milk, Butter & Cream                     8087
Spreads, Jam, Honey & Syrups             8000
Chips, Crisps & Salty Snacks             7611
Biscuits, Cookies & Wafers               7390
Juices & Nectars                         7004
Nuts, Seeds & Dried Fruit                6811
Bread & Savoury Bakery                   5271
Name: count, dtype: int64

### Takeaway
The cleaner removes ~17.8k duplicate/invalid Open Food Facts rows and 14.7k non-food BigBasket SKUs, derives sodium from salt, engineers a 28-bucket category from the product name (75.5% coverage), and keeps energy outliers *flagged* rather than deleted. See `reports/data_quality_report.md`.